In [11]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn

# Step 1: Data Preparation
# Load the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
column_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
]
data = pd.read_csv(url, names=column_names)

# Replace missing values marked by '?' with NaN
data.replace('?', np.nan, inplace=True)

# Drop rows with missing values
data.dropna(inplace=True)

# Convert columns to appropriate data types
data = data.astype(float)

# Convert the target variable to binary classification
data['target'] = data['target'].apply(lambda x: 1 if x > 0 else 0)

# Split the dataset into features and target
X = data.drop('target', axis=1)
y = data['target']

# Standardize the features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: Hypervector Generation
def generate_hypervector(size, seed=None):
    if seed is not None:
        np.random.seed(seed)
    return np.random.choice([-1, 1], size)

# Parameters
dim = 10000  # Dimension of hypervectors

# Generate random hypervectors for each feature
feature_hvs = [generate_hypervector(dim, seed=i) for i in range(X.shape[1])]

# Function to encode a data point into a hypervector
def encode_hypervector(data_point, feature_hvs):
    hypervector = np.zeros(dim)
    for i, value in enumerate(data_point):
        hypervector += value * feature_hvs[i]
    return np.sign(hypervector)

# Encode training and testing data
X_train_hv = np.array([encode_hypervector(x, feature_hvs) for x in X_train])
X_test_hv = np.array([encode_hypervector(x, feature_hvs) for x in X_test])

# Step 3: Model Training
class HDClassifier(nn.Module):
    def __init__(self, input_dim):
        super(HDClassifier, self).__init__()
        self.fc = nn.Linear(input_dim, 1)
    
    def forward(self, x):
        return torch.sigmoid(self.fc(x))

# Convert hypervectors to PyTorch tensors
X_train_hv_tensor = torch.tensor(X_train_hv, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_hv_tensor = torch.tensor(X_test_hv, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# Initialize the model, loss function, and optimizer
model = HDClassifier(dim)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_hv_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# Step 4: Evaluation
model.eval()
with torch.no_grad():
    y_pred_train = model(X_train_hv_tensor).round()
    y_pred_test = model(X_test_hv_tensor).round()
    
    train_accuracy = accuracy_score(y_train, y_pred_train)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    
    print(f'Training Accuracy: {train_accuracy:.4f}')
    print(f'Test Accuracy: {test_accuracy:.4f}')

Epoch [10/100], Loss: 0.4033
Epoch [20/100], Loss: 0.3480
Epoch [30/100], Loss: 0.3088
Epoch [40/100], Loss: 0.2713
Epoch [50/100], Loss: 0.2372
Epoch [60/100], Loss: 0.2088
Epoch [70/100], Loss: 0.1845
Epoch [80/100], Loss: 0.1635
Epoch [90/100], Loss: 0.1453
Epoch [100/100], Loss: 0.1296
Training Accuracy: 0.9873
Test Accuracy: 0.8833


In [12]:
data.target.value_counts()

target
0    160
1    137
Name: count, dtype: int64